Tools

Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairing of:

   1. A Schema, including the name of the tool, a description., and/or argument definitions(often a JSON schema)
   2. A function or coroutine to execute.

In [2]:
import os
from langchain.chat_models import init_chat_model
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")
model = init_chat_model("google_genai:gemini-3.5-flash-lite")
response = model.invoke("why do parrots talk?")
response

AIMessage(content=[{'type': 'text', 'text': 'Parrots talk primarily because they are **social learners** and **vocal mimics**. In the wild, talking (or rather, imitating sounds) is a crucial survival tool. \n\nHere is a breakdown of why parrots talk, both in the wild and in human homes:\n\n### 1. Social Bonding and Flock Integration\nIn the wild, parrots live in large, tight-knit flocks. To fit in, be accepted, and stay safe, a parrot needs to sound like the other members of its flock. They learn specific "dialects" or contact calls from their parents and flock mates. \n\n### 2. Claiming Territory and Mating\nWild parrots use vocalizations to establish territory, warn others of danger, and attract mates. They have exceptional memories and can recognize individual voices, helping them keep track of their mate and family members in a noisy rainforest.\n\n### 3. High Intelligence and Brain Structure\nParrots have evolved a specialized brain structure that allows for complex vocal learning

In [22]:
from langchain.tools import tool

@tool
def get_weather(location:str) ->str:
    """ Get the weather at a location"""
    return f"It's sunny in {location}"

model_with_tools = model.bind_tools([get_weather])

In [20]:
response = model_with_tools.invoke("What's the weather in Boston?")
response

AIMessage(content=[], additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"location": "Boston"}'}, '__gemini_function_call_thought_signatures__': {'call_5005285625210127631': 'EpYBCpMBAVjVB3umsS0QDCmYC7ZboY11tlU4oB1sgsp4vqU7zL9G94wRZfq9O0DuAti1eR3qznvgW4YpiNGgtL2kHMfSDMapPczb5sKMIOd76v4ece/H3TDjvw4l34/ZWuaCUgrU3ivDHhnepO0BK3WLUdEmyEc9FGPdb1TbFFcj/fqldqanxlYBQPYRHVzBH3eniTFYyZG9'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'normalized_finish_reason': 'tool_call', 'model_provider': 'google_genai'}, id='lc_run--01a21c38-8a0b-7291-b108-4a476b06c458-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'call_5005285625210127631', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 52, 'output_tokens': 16, 'total_tokens': 68, 'input_token_details': {'cache_read': 0}})

In [23]:
### Tool Execution Loops
## Step 1 : Model Generates tool calls
messages = [{"role":"user","content":"what's the weather in Boston"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

## Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)



## Step 3: Pass results back to model for final response
final_resonse = model_with_tools.invoke(messages)
print(final_resonse.text)


It's sunny in Boston


In [24]:
from langgraph.prebuilt import create_react_agent

# Automatically runs Steps 1, 2, and 3 in a loop until finished
agent = create_react_agent(model, tools=[get_weather])

result = agent.invoke({"messages": [("user", "What's the weather in Boston?")]})
print(result["messages"][-1].content)

C:\Users\Manas\AppData\Local\Temp\ipykernel_32584\214944810.py:4: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(model, tools=[get_weather])


[{'type': 'text', 'text': "It's sunny in Boston.", 'extras': {'signature': 'EpYBCpMBAVjVB3vCaRI+NkBtrhprLHQYm5hZNMI9hbcpXqWra+svznLcOnbUfVrYsbpIdBbDRSVh659qGxZgW8oShoNnal0PWRMlhlzqQnH2UXReUKK9U21KiLOQP9+ERqq+9WQQF4HmRSWYxbM7egl5j8RhvyDzT+9+osPu6RyHnZgq2WefPZm4iEfQ7DSkg1d/dodm55bs'}}]
